# Sector Dimension Loader

Maintains the `warehouse.dim_sector` dimension table with incremental refresh capability.

## Purpose
Track industry sectors and hierarchies from canonical metadata taxonomy.

## Key Features
* Stable surrogate keys for sectors (auto-increment)
* Sector hierarchies (primary/secondary levels)
* NAICS code mapping and keyword tagging
* SCD Type 1 (overwrite on change)
* Idempotent: safe to re-run

## Architecture
**Source**: Metadata CSV files (`/Workspace/.../LMIP/metadata/sectors.csv`)  
**Target**: `workspace.warehouse.dim_sector`  
**Metadata**: `workspace.metadata.dim_sector_refresh_log`  
**Mode**: Incremental (merge new/updated sectors)

## Batch Processing
* Tracks refresh history in metadata table
* Auto-assigns surrogate keys to new sectors
* Updates existing sectors with latest taxonomy
* Validates hierarchy and data quality after refresh

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")
dbutils.widgets.text("metadata_path", "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata", "Metadata Path")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"
METADATA_PATH = dbutils.widgets.get("metadata_path")

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_sector"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_sector_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create sector dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_sector (
  sector_sk BIGINT NOT NULL COMMENT 'Surrogate key for sector',
  sector_key STRING NOT NULL COMMENT 'Natural key from taxonomy',
  sector_name STRING NOT NULL COMMENT 'Sector display name',
  sector_category STRING NOT NULL COMMENT 'PRIMARY or SECONDARY',
  sector_level INT NOT NULL COMMENT 'Hierarchy level (1=primary, 2=secondary)',
  parent_sector STRING COMMENT 'Parent sector key',
  naics_code STRING COMMENT 'NAICS industry code',
  keywords ARRAY<STRING> COMMENT 'Keyword tags for matching',
  is_active BOOLEAN NOT NULL COMMENT 'Is sector active',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
  CONSTRAINT pk_dim_sector PRIMARY KEY (sector_sk)
)
USING DELTA
COMMENT 'Industry sector dimension with hierarchy';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_sector_refresh_log (
  run_id STRING,
  sectors_extracted INT,
  sectors_inserted INT,
  sectors_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks sector dimension refresh history';

In [0]:
print("Loading sector taxonomy from metadata files...", end=" ")

# Load sector taxonomy
sectors_df = spark.read.csv(
    f"{METADATA_PATH}/sectors.csv",
    header=True,
    inferSchema=True
)

# Transform for warehouse dimension
sector_extract_df = sectors_df.select(
    F.col("sector_key"),
    F.col("sector_name"),
    F.col("parent_sector"),
    F.col("naics_code"),
    F.when(F.col("parent_sector").isNull(), "PRIMARY").otherwise("SECONDARY").alias("sector_category"),
    F.when(F.col("parent_sector").isNull(), 1).otherwise(2).alias("sector_level"),
    F.lit(True).alias("is_active"),
    F.split(F.col("keywords"), "\\|").alias("keywords")
)

sectors_count = sector_extract_df.count()
print(f"✓ Loaded {sectors_count} sectors from taxonomy")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(sector_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("sectors_extracted", IntegerType(), True),
    StructField("sectors_inserted", IntegerType(), True),
    StructField("sectors_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing sectors into {TARGET_TABLE}...", end=" ")
    
    # Check if table schema matches expected schema FIRST
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['sector_sk', 'sector_key', 'sector_name', 'sector_category', 'sector_level', 'parent_sector', 'naics_code', 'keywords', 'is_active', 'updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    # Only query existing sectors if schema matches
    if schema_matches:
        existing_sectors = spark.sql(f"SELECT sector_key, sector_sk FROM {TARGET_TABLE}")
        
        # Join to assign keys (existing or new)
        from pyspark.sql.window import Window
        
        sectors_with_keys = sector_extract_df.alias("s").join(
            existing_sectors.alias("e"),
            F.col("s.sector_key") == F.col("e.sector_key"),
            "left"
        )
        
        # Assign surrogate keys
        window_spec = Window.orderBy("s.sector_key")
        
        sectors_final = sectors_with_keys.withColumn(
            "sector_sk",
            F.coalesce(F.col("e.sector_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            F.col("sector_sk").cast(LongType()),
            F.col("s.sector_key").alias("sector_key"),
            F.col("s.sector_name").alias("sector_name"),
            F.col("s.sector_category").alias("sector_category"),
            F.col("s.sector_level").alias("sector_level"),
            F.col("s.parent_sector").alias("parent_sector"),
            F.col("s.naics_code").alias("naics_code"),
            F.col("s.keywords").alias("keywords"),
            F.col("s.is_active").alias("is_active"),
            "updated_at"
        )
    else:
        # Schema mismatch: assign keys without joining to existing table
        from pyspark.sql.window import Window
        window_spec = Window.orderBy("sector_key")
        
        sectors_final = sector_extract_df.withColumn(
            "sector_sk",
            (F.lit(max_sk) + F.row_number().over(window_spec)).cast(LongType())
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            "sector_sk",
            "sector_key",
            "sector_name",
            "sector_category",
            "sector_level",
            "parent_sector",
            "naics_code",
            "keywords",
            "is_active",
            "updated_at"
        )
    
    # Create temp view for merge
    sectors_final.createOrReplaceTempView("sectors_to_merge")
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              sector_sk BIGINT NOT NULL COMMENT 'Surrogate key for sector',
              sector_key STRING NOT NULL COMMENT 'Natural key from taxonomy',
              sector_name STRING NOT NULL COMMENT 'Sector display name',
              sector_category STRING NOT NULL COMMENT 'PRIMARY or SECONDARY',
              sector_level INT NOT NULL COMMENT 'Hierarchy level (1=primary, 2=secondary)',
              parent_sector STRING COMMENT 'Parent sector key',
              naics_code STRING COMMENT 'NAICS industry code',
              keywords ARRAY<STRING> COMMENT 'Keyword tags for matching',
              is_active BOOLEAN NOT NULL COMMENT 'Is sector active',
              updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
              CONSTRAINT pk_dim_sector PRIMARY KEY (sector_sk)
            )
            USING DELTA
            COMMENT 'Industry sector dimension with hierarchy'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        sectors_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        sectors_inserted = sectors_final.count()
        sectors_updated = 0
        print(f"✓ Full refresh: {sectors_inserted} sectors inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING sectors_to_merge source
        ON target.sector_key = source.sector_key
        WHEN MATCHED THEN UPDATE SET
            target.sector_name = source.sector_name,
            target.sector_category = source.sector_category,
            target.sector_level = source.sector_level,
            target.parent_sector = source.parent_sector,
            target.naics_code = source.naics_code,
            target.keywords = source.keywords,
            target.is_active = source.is_active,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        sectors_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM sectors_to_merge
            WHERE sector_key NOT IN (SELECT sector_key FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        sectors_updated = sectors_count - sectors_inserted
        
        print(f"✓ Merge complete: {sectors_inserted} new, {sectors_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        sectors_count,
        sectors_inserted,
        sectors_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "sectors_extracted": sectors_count,
        "sectors_inserted": sectors_inserted,
        "sectors_updated": sectors_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        sectors_count if 'sectors_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate sector dimension and hierarchy
SELECT 
  COUNT(*) as total_sectors,
  COUNT(DISTINCT sector_category) as categories,
  COUNT(DISTINCT parent_sector) as parent_sectors,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_sectors,
  COUNT(DISTINCT CASE WHEN sector_level = 1 THEN sector_key END) as primary_sectors,
  COUNT(DISTINCT CASE WHEN sector_level = 2 THEN sector_key END) as secondary_sectors
FROM workspace.warehouse.dim_sector;

-- Show sector hierarchy
SELECT 
  sector_sk,
  sector_name,
  sector_category,
  sector_level,
  parent_sector,
  naics_code,
  is_active,
  updated_at
FROM workspace.warehouse.dim_sector
ORDER BY sector_category, sector_name
LIMIT 20;

-- Show refresh history
SELECT 
  run_id,
  sectors_extracted,
  sectors_inserted,
  sectors_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_sector_refresh_log
ORDER BY processed_at DESC
LIMIT 10;